# 01 · TF-IDF logistic regression

How well can we identify sarcasm from the comment alone? This baseline gives each word a TF-IDF weight and learns a logistic regression coefficient for it. The model returns a probability for label 1 (sarcastic).

Run notebook 00 first to create the shared 70/15/15 split. Install scikit-learn in the notebook's Python environment, then run these cells in order.

In [1]:
import csv
import json
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.pipeline import make_pipeline

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
data_dir = root / "data" / "processed"

def read_split(name):
    comments, labels, sample_ids = [], [], []
    with (data_dir / f"{name}.csv").open(encoding="utf-8", newline="") as source:
        for row in csv.DictReader(source):
            comments.append(row["comment"])
            labels.append(int(row["label"]))
            sample_ids.append(row["sample_id"])
    return comments, labels, sample_ids

## Train the baseline

Use individual words and cap the vocabulary at 50,000 terms to keep the sparse feature matrix manageable. Words must appear in at least two training comments. Keep common words like “not” because they can change the meaning of a comment.

In [2]:
train_comments, train_labels, _ = read_split("train")
baseline = make_pipeline(
    # Skip words seen in just one training comment and keep up to 50,000 terms.
    TfidfVectorizer(max_features=50_000, min_df=2),
    LogisticRegression(solver="liblinear", C=1.0, max_iter=200, random_state=522),
)
# Fit the vocabulary here so validation and test comments can't affect the word weights.
baseline.fit(train_comments, train_labels)
print(f"Training comments: {len(train_labels):,}")
print(f"Vocabulary size: {len(baseline[0].vocabulary_):,}")

Training comments: 666,295
Vocabulary size: 50,000


## Evaluate on validation and test

Use the same fitted model and a fixed probability threshold of 0.5 for both splits. Precision, recall, and F1 below refer to the sarcastic class. Majority-class accuracy shows what we'd get by always predicting the most common training label.

In [3]:
# This gives us a simple score to beat: guess the most common training label every time.
majority_label = int(sum(train_labels) > len(train_labels) / 2)
metrics = {}
prediction_dir = data_dir / "logistic_regression_predictions"
prediction_dir.mkdir(exist_ok=True)

for split in ("validation", "test"):
    comments, labels, sample_ids = read_split(split)
    # Column 1 gives the probability of sarcasm because our labels are 0 and 1.
    probabilities = baseline.predict_proba(comments)[:, 1]
    # Count a comment as sarcastic when its probability is at least 50%.
    predictions = (probabilities >= 0.5).astype(int)
    metrics[split] = {
        "rows": len(labels),
        "accuracy": accuracy_score(labels, predictions),
        "majority_accuracy": labels.count(majority_label) / len(labels),
        "precision": precision_score(labels, predictions, zero_division=0),
        "recall": recall_score(labels, predictions, zero_division=0),
        "f1": f1_score(labels, predictions, zero_division=0),
        "confusion_matrix": confusion_matrix(labels, predictions, labels=[0, 1]).tolist(),
    }
    # Keep the IDs so we can line these predictions up with the team's other models.
    with (prediction_dir / f"{split}.csv").open("w", encoding="utf-8", newline="") as target:
        writer = csv.writer(target)
        writer.writerow(("sample_id", "label", "prediction", "sarcasm_probability"))
        writer.writerows(zip(sample_ids, labels, predictions, probabilities))

    print(split, json.dumps(metrics[split], indent=2))

metrics_dir = root / "results" / "metrics"
metrics_dir.mkdir(parents=True, exist_ok=True)
(metrics_dir / "logistic_regression.json").write_text(json.dumps(metrics, indent=2) + "\n")

validation {
  "rows": 142777,
  "accuracy": 0.6914138831884687,
  "majority_accuracy": 0.5037576080180981,
  "precision": 0.7073671677332937,
  "recall": 0.6607994438651373,
  "f1": 0.6832908025734105,
  "confusion_matrix": [
    [
      51190,
      19662
    ],
    [
      24397,
      47528
    ]
  ]
}
test {
  "rows": 142777,
  "accuracy": 0.6907765256308789,
  "majority_accuracy": 0.5037576080180981,
  "precision": 0.7068408275122503,
  "recall": 0.6598262078554049,
  "f1": 0.682524844318525,
  "confusion_matrix": [
    [
      51169,
      19683
    ],
    [
      24467,
      47458
    ]
  ]
}


Read each confusion matrix by row (actual label) and column (predicted label), in the order 0, 1. The lower-left count is the number of sarcastic comments the model missed. The upper-right count is the number of non-sarcastic comments it flagged.